# STEP 1 — Load the dataset

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/raw/creditwise_enriched_synthetic_dataset.csv"
)

print("Shape:", df.shape)
df.head()

Shape: (12000, 44)


,applicant_id,gender,age,num_children,family_size,family_status,education_type,housing_type,own_car,own_property,...,monthly_income,disposable_income,credit_utilization_ratio,total_credit_exposure,income_band,dti_band,credit_history_band,employment_band,payment_risk_indicator,model_target_approval
0,100000,M,33,0,2,Married,Secondary,With parents,N,Y,...,5941.67,2312.26,0.1470,13616.55,High,Low,Developing,Experienced,Low,1
1,100001,F,42,3,4,Married,Secondary,Rented apartment,N,Y,...,2766.67,869.98,0.2888,17378.07,Medium,Moderate,Long,Early Career,Low,0
2,100002,F,48,0,1,Widow,Secondary,House / apartment,N,Y,...,5625.00,1305.56,0.2582,39941.43,High,Moderate,Developing,Experienced,Low,1
3,100003,F,31,0,1,Married,Higher education,With parents,Y,Y,...,3508.33,1133.80,0.1301,25271.82,Medium,Moderate,Established,Early Career,Low,0
4,100004,M,48,1,3,Separated,Incomplete higher,House / apartment,N,N,...,1666.67,1071.71,0.1626,1195.75,Low,Low,Long,Experienced,Low,1


# STEP 2 — Make a copy

In [4]:
df_clean = df.copy()

# STEP 3 — Check duplicate rows

In [5]:
df_clean.duplicated().sum()

np.int64(0)

In [6]:
df_clean["applicant_id"].duplicated().sum()

np.int64(0)

# STEP 4 — Check IDs

In [7]:
print("Total rows:", len(df_clean))
print("Unique applicant IDs:", df_clean["applicant_id"].nunique())

Total rows: 12000
Unique applicant IDs: 12000


# STEP 5 — Convert date

In [8]:
df_clean["application_date"] = pd.to_datetime(
    df_clean["application_date"],
    errors="coerce"
)

In [9]:
print(df_clean["application_date"].dtype)
print(df_clean["application_date"].isna().sum())

datetime64[ns]
0


# STEP 6 — Check numerical ranges

In [10]:
df_clean[[
    "age",
    "annual_income",
    "years_employed",
    "credit_history_months",
    "existing_credit_lines",
    "debt_to_income_ratio",
    "late_payments_24m",
    "requested_credit_limit",
    "monthly_debt_obligation",
    "monthly_expenses",
    "total_outstanding_debt",
    "current_credit_limit",
    "current_credit_balance",
    "credit_score",
    "monthly_income",
    "disposable_income",
    "credit_utilization_ratio",
    "total_credit_exposure"
]].describe().T

,count,mean,std,min,25%,50%,75%,max
age,12000.0,39.814250,10.508907,21.00,32.000000,40.0000,47.0000,70.0000
annual_income,12000.0,54987.883333,28553.415055,12000.00,35100.000000,49400.0000,68700.0000,255200.0000
years_employed,11649.0,5.258220,6.106028,0.00,0.800000,3.3000,7.6000,45.0000
credit_history_months,12000.0,73.243833,52.981495,0.00,33.000000,63.0000,104.0000,307.0000
existing_credit_lines,12000.0,1.404000,1.171565,0.00,1.000000,1.0000,2.0000,7.0000
debt_to_income_ratio,12000.0,0.228380,0.097501,0.00,0.162000,0.2270,0.2920,0.6150
late_payments_24m,12000.0,0.048667,0.321414,0.00,0.000000,0.0000,0.0000,6.0000
requested_credit_limit,12000.0,60893.230982,32937.231492,11144.40,30555.232500,57869.3300,86017.7475,145796.7500
monthly_debt_obligation,12000.0,1048.082446,757.126564,0.00,526.452500,878.2050,1360.6875,9645.6500
monthly_expenses,12000.0,2185.380037,1257.050305,305.00,1302.295000,1913.7200,2758.4025,12098.0800


# STEP 7 — Validate credit score

In [11]:
df_clean[
    (df_clean["credit_score"] < 300) |
    (df_clean["credit_score"] > 850)
].shape

(0, 44)

# STEP 8 — Validate DTI

In [12]:
df_clean["debt_to_income_ratio"].describe()

count    12000.000000
mean         0.228380
std          0.097501
min          0.000000
25%          0.162000
50%          0.227000
75%          0.292000
max          0.615000
Name: debt_to_income_ratio, dtype: float64

In [13]:
df_clean[
    (df_clean["debt_to_income_ratio"] < 0) |
    (df_clean["debt_to_income_ratio"] > 1)
].shape

(0, 44)

# STEP 9 — Validate credit utilization

In [14]:
df_clean["credit_utilization_ratio"].describe()

count    12000.000000
mean         0.222521
std          0.089516
min          0.020000
25%          0.161575
50%          0.219200
75%          0.278800
max          0.854400
Name: credit_utilization_ratio, dtype: float64

In [15]:
df_clean[
    (df_clean["credit_utilization_ratio"] < 0) |
    (df_clean["credit_utilization_ratio"] > 1)
].shape

(0, 44)

# STEP 10 — Validate target

In [16]:
df_clean["approved"].value_counts(dropna=False)

approved
0    7238
1    4762
Name: count, dtype: int64

In [17]:
set(df_clean["approved"].unique())

{np.int64(0), np.int64(1)}

# STEP 11 — Check categorical values

In [18]:
categorical_cols = [
    "gender",
    "family_status",
    "education_type",
    "housing_type",
    "own_car",
    "own_property",
    "income_type",
    "occupation_type",
    "product_type",
    "application_channel",
    "application_purpose"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df_clean[col].value_counts(dropna=False))


--- gender ---
gender
F    6179
M    5821
Name: count, dtype: int64

--- family_status ---
family_status
Married                 6977
Single / not married    2402
Separated               1086
Civil marriage          1004
Widow                    531
Name: count, dtype: int64

--- education_type ---
education_type
Secondary            5372
Higher education     3849
Incomplete higher    1482
Lower secondary       807
Academic degree       490
Name: count, dtype: int64

--- housing_type ---
housing_type
House / apartment      7725
Rented apartment       1649
With parents           1431
Municipal apartment     849
Co-op apartment         346
Name: count, dtype: int64

--- own_car ---
own_car
N      7060
Y      4806
NaN     134
Name: count, dtype: int64

--- own_property ---
own_property
Y    7439
N    4561
Name: count, dtype: int64

--- income_type ---
income_type
Working                 6259
Commercial associate    2615
Pensioner               1810
State servant           1093
Student   

# STEP 12 — Missing-value report

In [19]:
missing_report = (
    df_clean.isna()
    .sum()
    .reset_index()
)

missing_report.columns = ["column", "missing_count"]

missing_report["missing_percentage"] = (
    missing_report["missing_count"]
    / len(df_clean)
    * 100
)

missing_report = (
    missing_report[
        missing_report["missing_count"] > 0
    ]
    .sort_values("missing_percentage", ascending=False)
)

missing_report

,column,missing_count,missing_percentage
11,occupation_type,734,6.116667
13,years_employed,351,2.925000
8,own_car,134,1.116667


# STEP 13 — Check target leakage

In [26]:
(df_clean["approved"] == df_clean["model_target_approval"]).all()

np.True_

# STEP 14 — Identify metadata columns


In [21]:
metadata_cols = [
    "applicant_id",
    "synthetic_enrichment",
    "synthetic_version",
    "model_target_approval"
]

metadata_cols

['applicant_id',
 'synthetic_enrichment',
 'synthetic_version',
 'model_target_approval']

# STEP 15 — Create a data-quality summary

In [22]:
quality_summary = pd.DataFrame({
    "rows": [len(df_clean)],
    "columns": [df_clean.shape[1]],
    "duplicate_rows": [df_clean.duplicated().sum()],
    "duplicate_applicant_ids": [
        df_clean["applicant_id"].duplicated().sum()
    ],
    "missing_cells": [df_clean.isna().sum().sum()],
    "invalid_dates": [
        df_clean["application_date"].isna().sum()
    ]
})

quality_summary

,rows,columns,duplicate_rows,duplicate_applicant_ids,missing_cells,invalid_dates
0,12000,44,0,0,1219,0


In [27]:
df_clean["occupation_type"] = df_clean["occupation_type"].fillna("Unknown")
df_clean["own_car"] = df_clean["own_car"].fillna("Unknown")

In [28]:
df_clean["years_employed"] = (
    df_clean.groupby("income_type")["years_employed"]
    .transform(lambda x: x.fillna(x.median()))
)

In [29]:
df_clean.isna().sum().sort_values(ascending=False).head(10)

applicant_id      0
gender            0
age               0
num_children      0
family_size       0
family_status     0
education_type    0
housing_type      0
own_car           0
own_property      0
dtype: int64

In [30]:
df_clean.isna().sum().sum()

np.int64(0)

In [31]:
categorical_cols = [
    "gender",
    "family_status",
    "education_type",
    "housing_type",
    "own_car",
    "own_property",
    "income_type",
    "occupation_type",
    "product_type",
    "application_channel",
    "application_purpose",
    "income_band",
    "dti_band",
    "credit_history_band",
    "employment_band",
    "payment_risk_indicator",
    "synthetic_version"
]

for col in categorical_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

In [32]:
quality_checks = {
    "duplicate_rows": df_clean.duplicated().sum(),
    "duplicate_applicant_ids": df_clean["applicant_id"].duplicated().sum(),
    "missing_cells": df_clean.isna().sum().sum(),
    "invalid_credit_scores": (
        (df_clean["credit_score"] < 300) |
        (df_clean["credit_score"] > 850)
    ).sum(),
    "invalid_dti": (
        (df_clean["debt_to_income_ratio"] < 0) |
        (df_clean["debt_to_income_ratio"] > 1)
    ).sum(),
    "invalid_utilization": (
        (df_clean["credit_utilization_ratio"] < 0) |
        (df_clean["credit_utilization_ratio"] > 1)
    ).sum(),
    "invalid_target": (
        ~df_clean["approved"].isin([0, 1])
    ).sum()
}

quality_checks

{'duplicate_rows': np.int64(0),
 'duplicate_applicant_ids': np.int64(0),
 'missing_cells': np.int64(0),
 'invalid_credit_scores': np.int64(0),
 'invalid_dti': np.int64(0),
 'invalid_utilization': np.int64(0),
 'invalid_target': np.int64(0)}

# STEP 16 — Save the cleaned dataset

In [ ]:

df_clean.to_csv(
    "",
    index=False
)
print("Cleaned dataset save../data/processed/creditwise_cleaned.csvd successfully.")
print("Shape:", df_clean.shape)

Cleaned dataset saved successfully.
Shape: (12000, 44)


In [7]:
import pandas as pd
import joblib

df = pd.read_csv(
    "../data/processed/creditwise_cleaned.csv"
)

test_applicant = df.iloc[[0]].copy()

# Import the exact reusable feature function
from src.features.feature_pipeline import prepare_features

features = prepare_features(test_applicant)

# Load newly saved preprocessor
preprocessor = joblib.load(
    "../data/outputs/preprocessor.pkl"
)

processed = preprocessor.transform(features)

print("Input shape:", features.shape)
print("Processed shape:", processed.shape)
print("Expected features:", len(preprocessor.get_feature_names_out()))

Input shape: (1, 39)
Processed shape: (1, 77)
Expected features: 77


In [8]:
from src.models.scoring import score_applicant

prediction, probability, processed = score_applicant(
    test_applicant
)

print("Prediction:", prediction)
print("Probability:", probability)
print("Processed shape:", processed.shape)

Prediction: 1
Probability: 0.8161916450197011
Processed shape: (1, 77)
